In [1]:
import boto3
import json
bedrock = boto3.client('bedrock', region_name='us-east-2')
response = bedrock.list_foundation_models()
print(f"Available models: {len(response['modelSummaries'])}")
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-2')

Available models: 96


**Pipeline:** load `.txt` files → chunk (with overlap) → embed (Titan) → vector search (cosine) → re-rank (Claude) → answer grounded in the top chunks → Gradio UI.

In [2]:
# Extra dependencies (run once). Gradio needs a Python version that has wheels for its dependencies (3.10-3.13).
# %pip install numpy gradio
import re
import numpy as np
from pathlib import Path

DOCS_DIR = Path("docs")
EMBED_MODEL_ID = 'amazon.titan-embed-text-v2:0'
EMBED_DIMENSIONS = 512
LLM_MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# Sample knowledge base, only created if the folder is empty. Drop your own .txt files in ./docs to use them instead.
SAMPLE_DOCS = {
    "bedrock.txt": """AWS Bedrock is a fully managed service that offers foundation models from companies such as Anthropic, Amazon, Meta and Mistral through a single API.
You can call models with the InvokeModel API or the Converse API, and stream responses with InvokeModelWithResponseStream.

Bedrock Knowledge Bases provide a managed RAG workflow. Documents are stored in Amazon S3, split into chunks, converted into embeddings and indexed in a vector store such as OpenSearch Serverless. At query time the most relevant chunks are retrieved and passed to the model.

Bedrock Guardrails let you filter harmful content, block denied topics and redact personally identifiable information. Pricing is on-demand, per input and output token, or provisioned throughput for steady workloads.""",
    "rag.txt": """Retrieval Augmented Generation (RAG) improves answers by retrieving relevant information from a knowledge base and adding it to the prompt. This reduces hallucinations and lets a model use private or recent data without retraining.

A basic RAG pipeline has four steps: split documents into chunks, embed every chunk, retrieve the chunks closest to the question embedding, and generate an answer using those chunks as context.

Chunk size matters. Chunks that are too small lose context, while chunks that are too large dilute the embedding and waste prompt tokens. A small overlap between consecutive chunks avoids cutting an idea in half.

Re-ranking is an optional second stage. A fast vector search retrieves many candidates, then a more precise model scores each candidate against the question and keeps only the best few.""",
    "embeddings.txt": """An embedding is a vector of numbers that represents the meaning of a piece of text. Texts with similar meaning end up close together in the vector space, even when they share no words.

Closeness is usually measured with cosine similarity, which compares the angle between two vectors. If vectors are normalized to length one, cosine similarity is just a dot product.

Amazon Titan Text Embeddings V2 accepts up to 8,192 tokens of input and can return vectors with 256, 512 or 1024 dimensions. Smaller vectors are cheaper to store and search, at a small cost in accuracy.""",
}
DOCS_DIR.mkdir(exist_ok=True)
if not any(DOCS_DIR.glob("*.txt")):
    for name, text in SAMPLE_DOCS.items():
        (DOCS_DIR / name).write_text(text, encoding="utf-8")


# --- Task 1: load documents from text files ---
def load_documents(directory=DOCS_DIR):
    """Read every .txt file in `directory` -> [{'source': filename, 'text': content}]."""
    docs = []
    for path in sorted(Path(directory).glob("*.txt")):
        text = path.read_text(encoding="utf-8").strip()
        if text:
            docs.append({"source": path.name, "text": text})
    return docs


# --- Task 2: chunking for long documents ---
def chunk_text(text, chunk_size=500, overlap=100):
    """Sentence-aware chunking: pack sentences up to `chunk_size` chars; each new chunk starts with the
    last ~`overlap` chars worth of sentences from the previous one so ideas are not cut in half."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n{2,}', text) if s.strip()]
    chunks, current = [], []
    for sentence in sentences:
        if current and len(" ".join(current + [sentence])) > chunk_size:
            chunks.append(" ".join(current))
            tail = []
            for s in reversed(current):  # carry over the overlap
                if sum(len(x) + 1 for x in tail) + len(s) > overlap:
                    break
                tail.insert(0, s)
            current = tail
        current.append(sentence)
    if current:
        chunks.append(" ".join(current))
    return chunks


def build_chunks(docs, chunk_size=500, overlap=100):
    return [{"source": d["source"], "chunk_id": i, "text": c}
            for d in docs for i, c in enumerate(chunk_text(d["text"], chunk_size, overlap))]


docs = load_documents()
chunks = build_chunks(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")
print(f"Example chunk ({chunks[0]['source']}):\n{chunks[0]['text']}")

3 documents -> 6 chunks
Example chunk (bedrock.txt):
AWS Bedrock is a fully managed service that offers foundation models from companies such as Anthropic, Amazon, Meta and Mistral through a single API. You can call models with the InvokeModel API or the Converse API, and stream responses with InvokeModelWithResponseStream. Bedrock Knowledge Bases provide a managed RAG workflow. Documents are stored in Amazon S3, split into chunks, converted into embeddings and indexed in a vector store such as OpenSearch Serverless.


In [3]:
# --- Embeddings + vector store ---
def embed(text):
    """Titan embedding for one text, L2-normalized (so cosine similarity = dot product)."""
    body = json.dumps({"inputText": text, "dimensions": EMBED_DIMENSIONS, "normalize": True})
    response = bedrock_runtime.invoke_model(modelId=EMBED_MODEL_ID, body=body)
    return np.array(json.loads(response["body"].read())["embedding"], dtype=np.float32)


class VectorStore:
    """In-memory vector store with cosine-similarity search."""
    def __init__(self, embed_fn=embed):
        self.embed_fn, self.items, self.matrix = embed_fn, [], None

    def add(self, chunks):
        vectors = np.vstack([self.embed_fn(c["text"]) for c in chunks])
        self.items += chunks
        self.matrix = vectors if self.matrix is None else np.vstack([self.matrix, vectors])

    def search(self, query, k=5):
        q = self.embed_fn(query)
        q = q / np.linalg.norm(q)
        scores = self.matrix @ q
        top = np.argsort(scores)[::-1][:k]
        return [{**self.items[i], "score": float(scores[i])} for i in top]


store = VectorStore()
store.add(chunks)
print(f"Indexed {len(store.items)} chunks, vector shape {store.matrix.shape}")

for r in store.search("How do I make retrieval results more precise?", k=3):
    print(f"{r['score']:.3f}  [{r['source']}]  {r['text'][:90]}...")

Indexed 6 chunks, vector shape (6, 512)
0.283  [embeddings.txt]  Smaller vectors are cheaper to store and search, at a small cost in accuracy....
0.266  [rag.txt]  Retrieval Augmented Generation (RAG) improves answers by retrieving relevant information f...
0.206  [rag.txt]  Chunk size matters. Chunks that are too small lose context, while chunks that are too larg...


In [4]:
# --- Task 3: re-ranking of results ---
def claude(prompt, system=None, max_tokens=800):
    body = {"anthropic_version": "bedrock-2023-05-31", "max_tokens": max_tokens, "temperature": 0,
            "messages": [{"role": "user", "content": prompt}]}
    if system:
        body["system"] = system
    response = bedrock_runtime.invoke_model(modelId=LLM_MODEL_ID, body=json.dumps(body))
    return json.loads(response["body"].read())["content"][0]["text"]


def rerank(query, candidates, top_n=3):
    """Second stage: Claude scores every candidate 0-10 for relevance to the question; keep the best `top_n`.
    Falls back to the original vector order if the scores cannot be parsed."""
    numbered = "\n\n".join(f"[{i}] {c['text']}" for i, c in enumerate(candidates))
    prompt = (f"Question: {query}\n\nPassages:\n{numbered}\n\n"
              f"Rate how useful each passage is for answering the question, from 0 (irrelevant) to 10 (answers it directly). "
              f"Reply with ONLY a JSON list of {len(candidates)} numbers, in passage order.")
    try:
        match = re.search(r"\[.*?\]", claude(prompt, max_tokens=200), re.S)
        scores = json.loads(match.group(0))
        assert len(scores) == len(candidates)
    except Exception as e:
        print(f"Re-ranking failed ({e}); keeping vector order.")
        return candidates[:top_n]
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_n]
    return [{**c, "rerank_score": s} for c, s in ranked]


# --- Generation ---
def rag_answer(query, top_k=6, top_n=3, use_rerank=True):
    """Retrieve top_k by vector similarity, optionally re-rank down to top_n, answer from those chunks only."""
    candidates = store.search(query, k=top_k)
    context_chunks = rerank(query, candidates, top_n) if use_rerank else candidates[:top_n]
    context = "\n\n".join(f"[{i + 1}] (source: {c['source']})\n{c['text']}" for i, c in enumerate(context_chunks))
    answer = claude(
        f"Context:\n{context}\n\nQuestion: {query}",
        system="Answer the question using ONLY the provided context and cite passages like [1]. "
               "If the context does not contain the answer, say you don't know.")
    return answer, context_chunks


answer, used = rag_answer("What is re-ranking and why use it?")
print(answer)
print("\nSources:", [(c["source"], round(c.get("rerank_score", c["score"]), 2)) for c in used])

Based on the provided context:

Re-ranking is an optional second stage in a RAG pipeline that improves the quality of retrieved results. According to [1], "A fast vector search retrieves many candidates, then a more precise model scores each candidate against the question and keeps only the best few."

The reason to use re-ranking is to achieve better precision: the initial vector search is fast but may retrieve many candidates that aren't all equally relevant. The re-ranking stage uses a more precise model to score each candidate specifically against the question, filtering down to only the most relevant results before generating the final answer.

Sources: [('rag.txt', 10), ('rag.txt', 0), ('embeddings.txt', 0)]


In [5]:
# --- Task 4: simple web interface with Gradio ---
import gradio as gr

def ask(query, top_k, top_n, use_rerank):
    if not query.strip():
        return "Please enter a question.", ""
    answer, used = rag_answer(query, int(top_k), int(top_n), use_rerank)
    sources = "\n\n".join(
        f"**[{i + 1}] {c['source']}** (similarity {c['score']:.3f}"
        + (f", re-rank {c['rerank_score']}/10" if 'rerank_score' in c else "") + f")\n\n> {c['text']}"
        for i, c in enumerate(used))
    return answer, sources

with gr.Blocks(title="Basic RAG") as demo:
    gr.Markdown("# Basic RAG with Bedrock embeddings\nAsk a question about the documents in `./docs`.")
    question = gr.Textbox(label="Question", placeholder="What is chunk overlap for?")
    with gr.Row():
        top_k = gr.Slider(1, 10, value=6, step=1, label="Candidates retrieved (top_k)")
        top_n = gr.Slider(1, 5, value=3, step=1, label="Chunks sent to the model (top_n)")
        use_rerank = gr.Checkbox(value=True, label="Re-rank with Claude")
    button = gr.Button("Ask", variant="primary")
    answer_box = gr.Markdown(label="Answer")
    sources_box = gr.Markdown(label="Retrieved chunks")
    button.click(ask, [question, top_k, top_n, use_rerank], [answer_box, sources_box])
    question.submit(ask, [question, top_k, top_n, use_rerank], [answer_box, sources_box])

demo.launch(inline=True)

c:\Users\MatiasSarmiento\Desktop\Onboarding\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
